<div class="usecase-header-bar">
<div class="usecase-title">New Business Location</div>
</div>

<div class="usecase-details"><b>Authored by: </b>Steven Tuften</div>
<div class="usecase-details"><b>Date Created: </b>T2, 2024</div>
<div class="usecase-details"><b>Time to Complete:</b> 90 mins</div>
<div class="usecase-details"><b>Level: </b>Intermediate</div>
<div class="usecase-details"><b>Pre-requisite Skills: </b>Python</div>


<div id="scenario" class="usecase-unnumbered-heading"><b>Scenario</b></div>


<div class="usecase-body"><b>As a cafe, restaurant or bar owner</b>, I am seeking commercial space in the City of Melbourne to open a new venue or expand an existing one. I want to identify where similar businesses are located and compare venue seating with residential and employment density so I can explore potential business locations.</div>


<div id="what-this-teaches" class="usecase-unnumbered-heading"><b>What this usecase will teach you</b></div>


<div class="usecase-body">This usecase introduces CLUE data and demonstrates how to prepare and combine multiple datasets for interactive geospatial analysis.</div>


<div class="usecase-list">

- Understand what CLUE data is and how to access it.
- Explore datasets derived from the CLUE survey.
- Summarise residential, employment and venue seating data.
- Create choropleth and scatter-map visualisations with Plotly.
- Combine multiple map layers into one interactive business-location view.

</div>


<div id="background" class="usecase-unnumbered-heading"><b>Background</b></div>


<div class="usecase-body">The City of Melbourne conducts the Census of Land Use and Employment (CLUE), which captures information about land use, employment and economic activity. This usecase uses CLUE data to compare residential dwellings, employment and cafe/restaurant seating at block level as exploratory context for a new business location.</div>


<div class="usecase-datasets">

- [Residential dwellings](https://data.melbourne.vic.gov.au/explore/dataset/residential-dwellings/)
- [Blocks for Census of Land Use and Employment (CLUE)](https://data.melbourne.vic.gov.au/explore/dataset/blocks-for-census-of-land-use-and-employment-clue/)
- [Café, restaurant, bistro seats](https://data.melbourne.vic.gov.au/explore/dataset/cafes-and-restaurants-with-seating-capacity/)
- [Jobs per CLUE industry for blocks](https://data.melbourne.vic.gov.au/explore/dataset/employment-by-block-by-clue-industry/)

</div>


<div id="contents" class="usecase-unnumbered-heading"><b>Contents</b></div>


<div class="usecase-contents">

<div>1. <a href="#uc23-section-1">Data Loading and Examination</a></div>
<div>&nbsp;&nbsp;&nbsp;1.1. <a href="#uc23-sub-1-1">Required Libraries and Packages</a></div>
<div>&nbsp;&nbsp;&nbsp;1.2. <a href="#uc23-sub-1-2">Dataset Import through API</a></div>
<div>&nbsp;&nbsp;&nbsp;1.3. <a href="#uc23-sub-1-3">Fetch GeoJSON Dataset from the API</a></div>
<div>&nbsp;&nbsp;&nbsp;1.4. <a href="#uc23-sub-1-4">Fetch Residential Dwellings Dataset</a></div>
<div>&nbsp;&nbsp;&nbsp;1.5. <a href="#uc23-sub-1-5">Filter for the Year 2020</a></div>
<div>&nbsp;&nbsp;&nbsp;1.6. <a href="#uc23-sub-1-6">Retrieve and Display the Shape of the Dataset</a></div>
<div>&nbsp;&nbsp;&nbsp;1.7. <a href="#uc23-sub-1-7">Data Overview</a></div>

<div>2. <a href="#uc23-section-2">Summarising Residential Dwelling Data</a></div>
<div>3. <a href="#uc23-section-3">Visualising Residential Dwellings on a Choropleth Map</a></div>
<div>4. <a href="#uc23-section-4">Displaying the Residential Choropleth Map</a></div>
<div>5. <a href="#uc23-section-5">Visualising Residential Density and Venue Seating</a></div>
<div>6. <a href="#uc23-section-6">Plotting Residential Density and Venue Seating</a></div>
<div>7. <a href="#uc23-section-7">Building an Interactive Visualisation for a New Business Location</a></div>
<div>8. <a href="#uc23-section-8">Preparing Employment Density Data</a></div>
<div>9. <a href="#uc23-section-9">Plotting Employment Density</a></div>
<div>10. <a href="#uc23-section-10">Combining All Map Layers into One Interactive Visualisation</a></div>
<div>11. <a href="#uc23-section-11">Adding Interactive Layer Controls</a></div>

</div>


<div id="uc23-section-1" class="usecase-section-heading"><b>1. Data Loading and Examination</b></div>
<div class="usecase-body">Load the required Python libraries and retrieve the CLUE datasets used throughout the analysis.</div>


<div id="uc23-sub-1-1" class="usecase-sub-section-heading"><b>1.1. Required Libraries and Packages</b></div>
<div class="usecase-body">Import the packages needed for API requests, data preparation and interactive Plotly visualisations.</div>


In [1]:
import requests                   # For making HTTP requests
from io import StringIO           # For reading API CSV responses in memory
import pandas as pd               # For data manipulation and analysis
import plotly.graph_objs as go    # For detailed interactive plots
import plotly.express as px       # For interactive visualisations

<div id="uc23-sub-1-2" class="usecase-sub-section-heading"><b>1.2. Dataset Import through API</b></div>
<div class="usecase-body">Define a reusable function that downloads complete CSV datasets from the City of Melbourne Open Data API.</div>


In [2]:
# Function to collect a complete CSV dataset from the City of Melbourne Open Data API.
def API_Unlimited(dataset_id):
    base_url = "https://data.melbourne.vic.gov.au/api/explore/v2.1/catalog/datasets/"
    export_format = "csv"
    url = f"{base_url}{dataset_id}/exports/{export_format}"

    params = {
        "select": "*",
        "limit": -1,
        "lang": "en",
        "timezone": "UTC"
    }

    response = requests.get(url, params=params)
    response.raise_for_status()

    csv_content = response.content.decode("utf-8")
    dataset = pd.read_csv(StringIO(csv_content), delimiter=";")
    return dataset

<div class="usecase-body">

**Function summary: `API_Unlimited`**

The `API_Unlimited` function retrieves a complete dataset from the City of Melbourne Open Data API and returns it as a pandas DataFrame.

**Key points:**
- Fetches all available records from the selected dataset.
- Reads the CSV response directly in memory.
- Returns a pandas DataFrame for further analysis.
- Raises an HTTP error when the API request is unsuccessful.

</div>

<div id="uc23-sub-1-3" class="usecase-sub-section-heading"><b>1.3. Fetch GeoJSON Dataset from the API</b></div>
<div class="usecase-body">Define a helper that retrieves GeoJSON geometry for mapping CLUE blocks.</div>


In [3]:
def fetch_geojson_dataset_API(dataset_id):
    base_url = "https://data.melbourne.vic.gov.au/api/v2/catalog/datasets/"
    export_format = "geojson"
    url = f"{base_url}{dataset_id}/exports/{export_format}"

    response = requests.get(url)
    response.raise_for_status()

    # Plotly accepts a GeoJSON dictionary directly.
    return response.json()

<div class="usecase-body">

**Function summary: `fetch_geojson_dataset_API`**

The `fetch_geojson_dataset_API` function retrieves a dataset in **GeoJSON** format from the City of Melbourne Open Data API.

**Key points:**
- Fetches GeoJSON directly from the API.
- Returns a standard Python GeoJSON dictionary that can be passed to Plotly.
- Raises an HTTP error when the request is unsuccessful.

</div>

<div id="uc23-sub-1-4" class="usecase-sub-section-heading"><b>1.4. Fetch Residential Dwellings Dataset</b></div>
<div class="usecase-body">Retrieve the residential dwellings dataset through the API for subsequent block-level analysis.</div>


<div class="usecase-body">

In this code, the following actions are performed:

- **`dataset_id_1`**: The variable `dataset_id_1` is assigned the string `'residential-dwellings'`, which represents the dataset identifier for the "residential dwellings" dataset on the Melbourne Open Data platform.
  
- **`API_Unlimited(dataset_id_1)`**: The function `API_Unlimited()` is called with `dataset_id_1` as the argument. This function fetches the dataset named `'residential-dwellings'` from the Melbourne Open Data API and returns the data as a pandas DataFrame.

- **`res_dataset`**: The returned data from the `API_Unlimited()` function is stored in the variable `res_dataset`, which now contains the full dataset for further analysis.

**Summary**: 
The code snippet retrieves the "residential dwellings" dataset from the Melbourne Open Data platform using the `API_Unlimited` function and stores the data in the `res_dataset` variable for analysis.

</div>

In [4]:
dataset_id_1 = 'residential-dwellings'
res_dataset = API_Unlimited(dataset_id_1)


<div id="uc23-sub-1-5" class="usecase-sub-section-heading"><b>1.5. Filter for the Year 2020</b></div>
<div class="usecase-body">Restrict the residential dataset to 2020 and retain the fields required for this usecase.</div>


In [5]:
# Filter the residential dataset for 2020.
res_dataset = res_dataset[res_dataset["census_year"] == 2020].copy()

# Rename columns to match the naming used throughout this use case.
res_dataset.rename(
    columns={
        "property_id": "pbs_property_id",
        "base_property_id": "bps_base_id",
        "building_address": "street_name",
        "longitude": "x_coordinate",
        "latitude": "y_coordinate"
    },
    inplace=True
)

columns_to_keep = [
    "census_year",
    "block_id",
    "pbs_property_id",
    "bps_base_id",
    "street_name",
    "clue_small_area",
    "dwelling_type",
    "dwelling_number",
    "x_coordinate",
    "y_coordinate"
]
res_dataset = res_dataset[columns_to_keep]

<div class="usecase-body">

Next, we will look at one of the CLUE datasets to better understand its structure and how we can use it.

Our data requirements from this use case include the following:
- Number of Residential Dwellings per CLUE Block
- Number of Employees per CLUE Block
- Number of Seats (Indoor and Outdoor) per Venue and CLUE Block

For this exercise, we shall start by examining the Residential Dwelling dataset.
Each dataset in the Melbourne Open Data Portal has a unique identifier that can be used to retrieve the dataset through the City of Melbourne Open Data API.

This dataset is placed in a Pandas dataframe and we will inspect the first three rows.

</div>

<div id="uc23-sub-1-6" class="usecase-sub-section-heading"><b>1.6. Retrieve and Display the Shape of the Dataset</b></div>
<div class="usecase-body">Inspect the number of rows and columns and preview the residential records.</div>


In [6]:
# Retrieve the "CLUE Residential Dwellings 2020" dataset

print(f'The shape of dataset is {res_dataset.shape}.')
print('Below are the first few rows of this dataset:')

# Transpose the DataFrame for easier visual comparison.
res_dataset.head(3).T

The shape of dataset is (10404, 10).
Below are the first few rows of this dataset:


,0,1,2
census_year,2020,2020,2020
block_id,333,333,333
pbs_property_id,100121,106070,107735
bps_base_id,100121,106070,107735
street_name,210 Abbotsford Street NORTH MELBOURNE VIC 3051,3 Little Provost Street NORTH MELBOURNE VIC 3051,11 Provost Street NORTH MELBOURNE VIC 3051
clue_small_area,North Melbourne,North Melbourne,North Melbourne
dwelling_type,House/Townhouse,House/Townhouse,House/Townhouse
dwelling_number,1,1,1
x_coordinate,144.945938,144.947783,144.947368
y_coordinate,-37.802291,-37.802346,-37.802227


<div id="uc23-sub-1-7" class="usecase-sub-section-heading"><b>1.7. Data Overview</b></div>
<div class="usecase-body">Summarise the structure and relevant fields of the prepared residential dataset.</div>


<div class="usecase-body">

- **Dataset Size**: The dataset contains 10,403 records and 10 fields, each describing various attributes of individual residential properties.
  
- **Details of Each Record**:
  - The dataset provides the **number of dwellings** for each property along with the **type of dwelling**, such as House/Townhouse, Residential Apartments, etc.
  
- **Location Information**:
  - The location of each property is specified using:
    - **Latitude and Longitude**: Geographic coordinates to precisely locate the property.
    - **CLUE Small Area and Block ID**: Area-based identifiers used for the CLUE analysis.
    - **Property ID**: A unique identifier for each property.

- **Census Year**:
  - The **Census year** is included in the dataset, showing when the data was collected. For this analysis, it focuses on the **2020 CLUE Census**.

- **Analysis Scope**:
  - For our analysis of this dataset and others, we will be restricting the analysis to the **2020 CLUE Census** and summarising the data at the **CLUE Block level**.

</div>

<div id="uc23-section-2" class="usecase-section-heading"><b>2. Summarising Residential Dwelling Data</b></div>
<div class="usecase-body">Aggregate individual residential records to CLUE block level so dwelling density can be mapped.</div>


<div class="usecase-body">

We want to plot the density of both residential dwellings and employment at city block level rather than a specific property or address. We can use a __[choropleth map](https://en.wikipedia.org/wiki/Choropleth_map)__ to do this.

Let's start by summarising the data at CLUE small area and Block level.

**Note:** We include CLUE Small Area as one of our group by fields so we can display the CLUE Small area name in the popup window when you hover over the area on the map.

We want to summarise the data by summing the number of dwellings across all rows in the same CLUE Block.

The following cell creates a dataframe containing this summary of residential dwellings.

</div>

<div class="usecase-body">

This code processes the **CLUE Residential Dwellings 2020** dataset by ensuring proper data types and creating an aggregated dataset based on the number of dwellings per block:

- **Casting Data Types**: 
  - Columns such as `census_year` and `dwelling_number` are cast to **integer**, while `x_coordinate` and `y_coordinate` (latitude and longitude) are cast to **float** to allow accurate numerical and geographic operations.
  - Remaining columns are converted to their optimal types using `convert_dtypes()`.

- **Aggregation**:
  - The dataset is grouped by `block_id` and `clue_small_area` to calculate the **total number of dwellings** for each block using the `dwelling_number` field.
  - This aggregated dataset shows the sum of dwellings per block, allowing for a more granular analysis of residential distribution.

- **Flattening Grouped Columns**:
  - After the group-by operation, column headers are flattened to simplify their structure.
  - The columns `clue_small_area` and `dwelling_numbersum` are renamed to **`clue_area`** and **`dwelling_count`** respectively for clarity.

**Output**:
The resulting dataset provides a summarised view of the total number of dwellings per block and clue area, ready for visualisation and further analysis.

</div>

In [7]:
# Cast data types so the dataset can be summarised correctly.
res_dataset[["census_year", "dwelling_number"]] = (
    res_dataset[["census_year", "dwelling_number"]].astype(int)
)
res_dataset[["x_coordinate", "y_coordinate"]] = (
    res_dataset[["x_coordinate", "y_coordinate"]].astype(float)
)
res_dataset = res_dataset.convert_dtypes()

# Create the aggregate dataset.
group_by_fields = ["block_id", "clue_small_area"]
aggregate_fields = {"dwelling_number": ["sum"]}

dwellingsByBlock = pd.DataFrame(
    res_dataset.groupby(group_by_fields, as_index=False).agg(aggregate_fields)
)

# A pandas group-by creates two heading levels. Flatten them for easier plotting.
dwellingsByBlock.columns = dwellingsByBlock.columns.map("".join)
dwellingsByBlock.rename(
    columns={
        "clue_small_area": "clue_area",
        "dwelling_numbersum": "dwelling_count"
    },
    inplace=True
)

# Keep block identifiers consistent with the GeoJSON feature identifiers.
dwellingsByBlock["block_id"] = dwellingsByBlock["block_id"].astype(str)

dwellingsByBlock.head(5)

,block_id,clue_area,dwelling_count
0,1,Melbourne (CBD),385
1,11,Melbourne (CBD),690
2,12,Melbourne (CBD),190
3,13,Melbourne (CBD),112
4,14,Melbourne (CBD),99


<div id="uc23-section-3" class="usecase-section-heading"><b>3. Visualising Residential Dwellings on a Choropleth Map</b></div>
<div class="usecase-body">Retrieve CLUE block geometry and prepare it to match the residential block identifiers.</div>


<div class="usecase-body">

We use the __[Plotly Python Open Source Graphing Library](https://plotly.com/python/)__ to generate maps from __[mapbox](https://www.mapbox.com/)__.

Creating a choropleth map requires us to know the geometry(shape) of each CLUE Block area as a collection of latitude and longitude points defining a polygon. This data can be downloaded from the Melbourne Open Data Portal in __[GeoJSON](https://en.wikipedia.org/wiki/GeoJSON)__ format.

We also need to supply the data to be used to highlight the CLUE Blocks and that data must include the same unique identifier for each Block contained in the GeoJSON data set.

Below we extract the Melbourne CLUE Block polygons into a GeoJSON datatype.

</div>

In [8]:
dataset_id_2 = "blocks-for-census-of-land-use-and-employment-clue"
block = fetch_geojson_dataset_API(dataset_id_2)

# Plotly matches `locations` against this GeoJSON property.
feature_id_key = "properties.block_id"

# Normalise GeoJSON block IDs to strings so they match the plotting DataFrames.
for feature in block.get("features", []):
    if "block_id" in feature.get("properties", {}):
        feature["properties"]["block_id"] = str(feature["properties"]["block_id"])

block

{'type': 'FeatureCollection',
 'features': [{'type': 'Feature',
   'geometry': {'coordinates': [[[144.9636195556313, -37.8024996413744],
      [144.9637542392081, -37.80170947126356],
      [144.9622802082444, -37.80154977428353],
      [144.962143619863, -37.80233353916553],
      [144.9620040647283, -37.80312235578667],
      [144.9634851596905, -37.80328625382616],
      [144.9636195556313, -37.8024996413744]]],
    'type': 'Polygon'},
   'properties': {'geo_point_2d': {'lon': 144.96288135741216,
     'lat': -37.80241770440398},
    'block_id': '244',
    'region_name': 'Carlton',
    'area_name': 'Carlton'}},
  {'type': 'Feature',
   'geometry': {'coordinates': [[[144.9637542392081, -37.80170947126356],
      [144.9636195556313, -37.8024996413744],
      [144.9650905737863, -37.80265939405505],
      [144.9652158883625, -37.80193578876639],
      [144.9654504040578, -37.80058158000915],
      [144.9639698423439, -37.80042189269609],
      [144.9637542392081, -37.80170947126356]]],


<div id="uc23-section-4" class="usecase-section-heading"><b>4. Displaying the Residential Choropleth Map</b></div>
<div class="usecase-body">Plot residential dwelling density by CLUE block using an interactive Mapbox choropleth.</div>


<div class="usecase-body">

Now using just one function call called 'choropleth_mapbox' we can display an interactive map using the **block** GeoJSON data to define the regions and the **dwellingsByBlock** dataframe to define the summarised data by block.

</div>

In [9]:
fig = px.choropleth_mapbox(dwellingsByBlock, 
                           geojson=block, 
                           locations='block_id', 
                           color='dwelling_count', 
                           color_continuous_scale=["#ECFDF5", "#059669"], 
                           range_color=(0, dwellingsByBlock['dwelling_count'].max()), 
                           featureidkey=feature_id_key, 
                           mapbox_style="open-street-map",  # Changed map style to a simpler one
                           zoom=12.15, 
                           center={"lat": -37.813, "lon": 144.945}, 
                           opacity=0.5, 
                           hover_name='clue_area', 
                           hover_data={'block_id':True,'dwelling_count':True}, 
                           labels={'dwelling_count':'Number of Dwellings','block_id':'CLUE Block Id'}, 
                           title='Residential Dwellings by CLUE Block Id for 2020', 
                           width=950, height=800 
                          )
fig.show()


C:\Users\SHARP\AppData\Local\Temp\ipykernel_19908\533064510.py:1: DeprecationWarning: *choropleth_mapbox* is deprecated! Use *choropleth_map* instead. Learn more at: https://plotly.com/python/mapbox-to-maplibre/
  fig = px.choropleth_mapbox(dwellingsByBlock,


<div class="usecase-body">

You've successfully used Melbourne CLUE Open Data and Plotly to visualise residential density in the City of Melbourne!<br>
Now zoom in and out on the map above to explore the city and areas of high and low residential density.<br><br>
This is your first step to selecting a suitable location for your new business!

</div>

<div class="usecase-body">

__You can explore the Residential Density data [Click here](../dataanalysis/eda-clue-residentialdwellings.ipynb)__.

</div>

<div id="uc23-section-5" class="usecase-section-heading"><b>5. Visualising Residential Density and Venue Seating</b></div>
<div class="usecase-body">Prepare cafe and restaurant seating data so it can be compared with residential density.</div>


<div class="usecase-body">

To build our view of cafe venue seating and how it relates to residential density we need to visualise both datasets on the same interactive map view.

We can do this by adding a new layer (or "trace" as it is called in Plotly) to our previous map of residential density.

Let's extract the Melbourne CLUE cafe, restaurant, bistro seats dataset and summarise it so its ready to plot.

</div>

In [10]:
# Pull the cafe, restaurant and bistro seating dataset.
dataset_id_3 = "cafes-and-restaurants-with-seating-capacity"
cafe_dataset = API_Unlimited(dataset_id_3)

# Filter the dataset for 2020.
cafe_dataset = cafe_dataset[cafe_dataset["census_year"] == 2020].copy()

# Cast columns to the required data types.
cafe_dataset.rename(
    columns={"longitude": "x_coordinate", "latitude": "y_coordinate"},
    inplace=True
)

integer_columns = [
    "census_year",
    "block_id",
    "property_id",
    "base_property_id",
    "industry_anzsic4_code",
    "number_of_seats"
]
float_columns = ["x_coordinate", "y_coordinate"]

cafe_dataset[integer_columns] = cafe_dataset[integer_columns].astype(int)
cafe_dataset[float_columns] = cafe_dataset[float_columns].astype(float)
cafe_dataset = cafe_dataset.convert_dtypes()

# Summarise venue seating by location.
group_by_fields = ["clue_small_area", "block_id", "y_coordinate", "x_coordinate"]
aggregate_fields = {"number_of_seats": ["sum"]}

seatsByLocn = pd.DataFrame(
    cafe_dataset.groupby(group_by_fields, as_index=False).agg(aggregate_fields)
)
seatsByLocn.columns = seatsByLocn.columns.map("".join)
seatsByLocn.rename(
    columns={
        "clue_small_area": "clue_area",
        "number_of_seatssum": "number_of_seats"
    },
    inplace=True
)

seatsByLocn["block_id"] = seatsByLocn["block_id"].astype(str)
seatsByLocn["number_of_seats"] = seatsByLocn["number_of_seats"].astype(int)

# Scale bubble sizes into 16 steps while keeping small venues visible.
seat_range = seatsByLocn["number_of_seats"].max() - seatsByLocn["number_of_seats"].min()
scale_interval = seat_range / 16 if seat_range else 1

seatsByLocn["scale"] = (
    (seatsByLocn["number_of_seats"] - seatsByLocn["number_of_seats"].min())
    / scale_interval
    + 3
).astype(int)

seatsByLocn.head(10)

,clue_area,block_id,y_coordinate,x_coordinate,number_of_seats,scale
0,Carlton,203,-37.796707,144.965534,51,3
1,Carlton,203,-37.79668,144.9649,42,3
2,Carlton,204,-37.797808,144.965164,50,3
3,Carlton,204,-37.797255,144.965754,120,3
4,Carlton,205,-37.799441,144.964854,96,3
5,Carlton,205,-37.798999,144.964765,80,3
6,Carlton,205,-37.798721,144.965258,41,3
7,Carlton,206,-37.800492,144.96659,51,3
8,Carlton,206,-37.80019,144.966717,140,3
9,Carlton,206,-37.800046,144.966741,115,3


<div class="usecase-body">

Above we can see our summary dataframe has calculated the total number of seats (indoor and outdoor) at each unique location (latitude and longitude).

Since there is such a wide variance in venue seating across the city we need to scale the size of the bubbles drawn on the map to just a few (16) distinct sizes.

We set the lowest scale to 3 to ensure even the smallest venue's bubble is large enough when one zooms in at block level.

The next step is to display both the Choropleth and Scatter maps.
We first draw the choropleth map showing residential density.
We then draw the scatter plot assigning it as a trace (aka "layer") to the existing figure then show both.

</div>

<div id="uc23-section-6" class="usecase-section-heading"><b>6. Plotting Residential Density and Venue Seating</b></div>
<div class="usecase-body">Overlay venue seating markers on the residential-density choropleth to compare demand context and existing capacity.</div>


In [11]:
# Plot residential density and venue seating
fig = px.choropleth_mapbox(dwellingsByBlock, geojson=block, locations='block_id', color='dwelling_count',
                           color_continuous_scale=["#ECFDF5", "#059669"],
                           range_color=(0, dwellingsByBlock['dwelling_count'].max()),
                           featureidkey=feature_id_key,
                           mapbox_style="open-street-map",  # Changed to open-street-map
                           zoom=12.15,
                           center = {"lat": -37.813, "lon": 144.945},
                           opacity=0.5,
                           hover_name='clue_area',
                           hover_data={'block_id':True,'dwelling_count':True},
                           labels={'dwelling_count':'Number of Dwellings','block_id':'CLUE Block Id'},
                           title='Residential Dwellings Density & Venue Seating (2020)',
                           width=950, height=800
                          )

# Plot of venue seating
fig2 = px.scatter_mapbox(seatsByLocn, lat="y_coordinate", lon="x_coordinate", size="scale",
                        mapbox_style="open-street-map",  # Changed to open-street-map
                        zoom=12.15,
                        center = {"lat": -37.813, "lon": 144.945},
                        opacity=0.70,
                        hover_name="clue_area",
                        hover_data={"block_id":True,"scale":False,"number_of_seats":True,"x_coordinate":False,"y_coordinate":False},
                        color_discrete_sequence=["#9333EA"],
                        labels={'number_of_seats':'Number of Seats', 'block_id':'CLUE Block Id'},
                        width=950, height=800)

# Add the venue seating layer to the residential density map
fig.add_trace(fig2.data[0])

# Show the plot
fig.show()

C:\Users\SHARP\AppData\Local\Temp\ipykernel_19908\2008823968.py:2: DeprecationWarning: *choropleth_mapbox* is deprecated! Use *choropleth_map* instead. Learn more at: https://plotly.com/python/mapbox-to-maplibre/
  fig = px.choropleth_mapbox(dwellingsByBlock, geojson=block, locations='block_id', color='dwelling_count',
C:\Users\SHARP\AppData\Local\Temp\ipykernel_19908\2008823968.py:18: DeprecationWarning: *scatter_mapbox* is deprecated! Use *scatter_map* instead. Learn more at: https://plotly.com/python/mapbox-to-maplibre/
  fig2 = px.scatter_mapbox(seatsByLocn, lat="y_coordinate", lon="x_coordinate", size="scale",


<div class="usecase-body">

You've successfully used Melbourne CLUE Open Data and Plotly to visualise residential density and venue seating in the City of Melbourne in one map!<br>
Now zoom in and out on the map above to explore the city and areas of high residential density but low venue seating.<br><br>
This could be a possible location for your new business!

</div>

<div class="usecase-body">

__You can explore the Venue Seating data in more detail[Click here](../dataanalysis/eda-clue-venueseats.ipynb)__.

</div>

<div id="uc23-section-7" class="usecase-section-heading"><b>7. Building an Interactive Visualisation for a New Business Location</b></div>
<div class="usecase-body">Extend the analysis by adding employment density and preparing all three layers for a combined interactive view.</div>


<div id="uc23-section-8" class="usecase-section-heading"><b>8. Preparing Employment Density Data</b></div>
<div class="usecase-body">Retrieve and prepare employment-by-block data for comparison with residential density and venue seating.</div>


<div class="usecase-body">

In the previous step, we added venue seating as a separate trace over the residential-density map.

We now add employment density to the analysis. Because residential density and employment density are both choropleth layers at CLUE block level, the final visualisation allows the user to switch between them while keeping venue seating available as an overlay.

The next cell retrieves the **Employment by Block by CLUE Industry** dataset and prepares the fields required for plotting.

</div>

In [12]:
# Pull the employment-by-block dataset.
dataset_id_4 = "employment-by-block-by-clue-industry"
jobs_dataset = API_Unlimited(dataset_id_4)

# Filter the dataset for 2020.
jobs_dataset = jobs_dataset[jobs_dataset["census_year"] == 2020].copy()

# Rename and retain only the columns required for this analysis.
jobs_dataset.rename(
    columns={"total_jobs_in_block": "total_employment_in_block"},
    inplace=True
)
columns_to_keep = ["clue_small_area", "block_id", "total_employment_in_block"]
employmentByBlock = jobs_dataset[columns_to_keep].copy()

employmentByBlock.rename(
    columns={"clue_small_area": "clue_area"},
    inplace=True
)

# Replace missing values with zero and cast to suitable data types.
employmentByBlock.fillna(0, inplace=True)
employmentByBlock[["block_id", "total_employment_in_block"]] = (
    employmentByBlock[["block_id", "total_employment_in_block"]].astype(int)
)

# Exclude the City of Melbourne summary row.
employmentByBlock = employmentByBlock[employmentByBlock["block_id"] > 0].copy()

# Keep block identifiers consistent with the GeoJSON feature identifiers.
employmentByBlock["block_id"] = employmentByBlock["block_id"].astype(str)
employmentByBlock = employmentByBlock.convert_dtypes()

employmentByBlock.head(5)

,clue_area,block_id,total_employment_in_block
764,Melbourne (CBD),1,764
765,Melbourne (CBD),21,4892
766,Melbourne (CBD),22,5173
767,Melbourne (CBD),24,7380
768,Melbourne (CBD),25,3457


<div class="usecase-body">

Now we have a dataset showing total number of employees by CLUE block, let's visualise it as a choropleth map and overlay venue seating.

In this map visualisation we will use a different map style called "open-street-map" which lets us identify the names of venues close to where the venue seating measures have been reported. **Note that not all venues may have been marked on Open Street Maps.**

Mapbox styles which do not require a Mapbox API token are 'open-street-map', 'white-bg', 'carto-positron', 'carto-darkmatter', 'stamen- terrain', 'stamen-toner', 'stamen-watercolor'. Mapbox styles which do require a Mapbox API token are 'basic', 'streets', 'outdoors', 'light', 'dark', 'satellite', 'satellite- streets'.

**Source:** __[plotly.express.line_mapbox documentation](https://plotly.com/python-api-reference/generated/plotly.express.line_mapbox.html)__

</div>

<div id="uc23-section-9" class="usecase-section-heading"><b>9. Plotting Employment Density</b></div>
<div class="usecase-body">Display employment density by CLUE block and overlay venue seating for comparison.</div>


In [13]:
fig = px.choropleth_mapbox(employmentByBlock, geojson=block, locations='block_id', color='total_employment_in_block',
                           color_continuous_scale=["#EFF6FF", "#2563EB"],
                           range_color=(0, employmentByBlock['total_employment_in_block'].max()),
                           featureidkey=feature_id_key,
                           mapbox_style="open-street-map",
                           zoom=12.15,
                           center = {"lat": -37.813, "lon": 144.945},
                           opacity=0.5,
                           hover_name='clue_area',
                           hover_data={'block_id':True,'total_employment_in_block':True},
                           labels={'total_employment_in_block':'Number of Employees','block_id':'CLUE Block Id'},
                           title='Employment Density & Venue Seating (2020)',
                           width=950, height=800
                          )

# Plot of venue seating
fig2 = px.scatter_mapbox(seatsByLocn, lat="y_coordinate", lon="x_coordinate", size="scale",
                        mapbox_style="open-street-map",
                        zoom=12.15,
                        center = {"lat": -37.813, "lon": 144.945},
                        opacity=0.70,
                        hover_name="clue_area",
                        hover_data={"block_id":True,"scale":False,"number_of_seats":True,"x_coordinate":False,"y_coordinate":False},
                        color_discrete_sequence=["#9333EA"],
                        labels={'number_of_seats':'Number of Seats', 'block_id':'CLUE Block Id'},
                        width=950, height=800)
fig.add_trace(fig2.data[0])

fig.show()

C:\Users\SHARP\AppData\Local\Temp\ipykernel_19908\2640116972.py:1: DeprecationWarning: *choropleth_mapbox* is deprecated! Use *choropleth_map* instead. Learn more at: https://plotly.com/python/mapbox-to-maplibre/
  fig = px.choropleth_mapbox(employmentByBlock, geojson=block, locations='block_id', color='total_employment_in_block',
C:\Users\SHARP\AppData\Local\Temp\ipykernel_19908\2640116972.py:17: DeprecationWarning: *scatter_mapbox* is deprecated! Use *scatter_map* instead. Learn more at: https://plotly.com/python/mapbox-to-maplibre/
  fig2 = px.scatter_mapbox(seatsByLocn, lat="y_coordinate", lon="x_coordinate", size="scale",


<div id="uc23-section-10" class="usecase-section-heading"><b>10. Combining All Map Layers into One Interactive Visualisation</b></div>
<div class="usecase-body">Create a single Plotly figure containing residential density, employment density and venue seating layers.</div>


<div class="usecase-body">

Let's now build a single Mapbox visualisation using our three datasets.

Our first step is to create a base plotly figure to which we can add each individual map plot as a new layer.

The title of the visualisation and any common parameters can be set using the fig.update_layout() method.

In the cell below we also have defined two custom colorscales, one continuous for the choropleth map and the other discrete for the scatter map plot.

We then create a figure for each dataset and add it as a layer to the base figure using the fig.add_trace() method.

</div>

In [14]:
# Define custom colour scale for choropleth (continuous) and scatter (discrete)
custom_continuous_colorscale = [(0, "#ECFDF5"), (1, "#059669")]
custom_discrete_colorscale = ["#9333EA"]

# Create the base figure to which layers(traces) will be added.
fig = go.Figure()

# Set the default style for the map
fig.update_layout(mapbox_style="open-street-map")
fig.update_layout(hovermode='closest')
fig.update_layout(mapbox_center_lat=-37.813, mapbox_center_lon=144.945, mapbox_zoom=12.15)
fig.update_layout(width=950, height=800)
fig.update_layout(title='Residential & Employment Density plus Venue Seating (2020)')
fig.update_layout(coloraxis_colorscale=custom_continuous_colorscale)
fig.update_layout(coloraxis_colorbar={'title':'Density'})

# Create the definition for the Residential Dwellings Layer
fig1 = px.choropleth_mapbox(dwellingsByBlock, geojson=block, locations='block_id', color='dwelling_count',
                           range_color=(0, dwellingsByBlock['dwelling_count'].max()),
                           featureidkey=feature_id_key,
                           hover_name='clue_area',
                           hover_data={'block_id':True,'dwelling_count':True},
                           labels={'dwelling_count':'Number of Dwellings','block_id':'CLUE Block Id'},
                           opacity=0.5,

                          )
fig.add_trace(fig1.data[0]) # add this layer to the base figure

# Create the definition for the Employment Layer
fig2 = px.choropleth_mapbox(employmentByBlock, geojson=block, locations='block_id', color='total_employment_in_block',
                           range_color=(0, employmentByBlock['total_employment_in_block'].max()),
                           featureidkey=feature_id_key,
                           hover_name='clue_area',
                           hover_data={'block_id':True,'total_employment_in_block':True},
                           labels={'total_employment_in_block':'Number of Employees','block_id':'CLUE Block Id'},
                           opacity=0.5
                          )
fig.add_trace(fig2.data[0]) # add this layer to the base figure

# Create the definition for the Venue Seating Layer
fig3 = px.scatter_mapbox(seatsByLocn, lat="y_coordinate", lon="x_coordinate", size="scale",
                        hover_name="clue_area",
                        hover_data={"block_id":True,"scale":False,"number_of_seats":True,"x_coordinate":False,"y_coordinate":False},
                        labels={'number_of_seats':'Number of Seats', 'block_id':'CLUE Block Id'},
                        opacity=0.70, color_discrete_sequence=custom_discrete_colorscale
                        )
fig.add_trace(fig3.data[0]) # add this layer to the base figure

C:\Users\SHARP\AppData\Local\Temp\ipykernel_19908\1576370097.py:18: DeprecationWarning: *choropleth_mapbox* is deprecated! Use *choropleth_map* instead. Learn more at: https://plotly.com/python/mapbox-to-maplibre/
  fig1 = px.choropleth_mapbox(dwellingsByBlock, geojson=block, locations='block_id', color='dwelling_count',
C:\Users\SHARP\AppData\Local\Temp\ipykernel_19908\1576370097.py:30: DeprecationWarning: *choropleth_mapbox* is deprecated! Use *choropleth_map* instead. Learn more at: https://plotly.com/python/mapbox-to-maplibre/
  fig2 = px.choropleth_mapbox(employmentByBlock, geojson=block, locations='block_id', color='total_employment_in_block',
C:\Users\SHARP\AppData\Local\Temp\ipykernel_19908\1576370097.py:41: DeprecationWarning: *scatter_mapbox* is deprecated! Use *scatter_map* instead. Learn more at: https://plotly.com/python/mapbox-to-maplibre/
  fig3 = px.scatter_mapbox(seatsByLocn, lat="y_coordinate", lon="x_coordinate", size="scale",


<div class="usecase-body">

Finally, we define buttons and text to appear along the top of the map.

Each button turns on a combination of layers when it is clicked. The layers it turns on are defined in the 'visible' arg array with the order of boolean values corresponding to the map layers in the order they were added.

For example: When the 'Residential Density & Seating' button is clicked it turns on the 1st and 3rd layer as defined by the following argument 'visible':[True, False, True] . The 1st layer was the Residential Dwelling density choropleth map and the 3rd layer was the Venue Seating Scatter map.

</div>

<div id="uc23-section-11" class="usecase-section-heading"><b>11. Adding Interactive Layer Controls</b></div>
<div class="usecase-body">Add a dropdown control so the user can switch between venue seating, residential density and employment density views.</div>


In [15]:
# Turn off all choropleth layers
fig.update_traces(visible=False, selector=dict(type='choroplethmapbox'))

# Add buttons for selection on plot
buttons = [dict(method='update',
                label='Venue Seating only',  visible=True,
                args=[{'label': 'Venue Seating', 'visible':[False, False, True]}]),
           dict(method='update',
                label='Residential Density & Seating', visible=True,
                args=[{'label': 'Residential Dwelling Density','visible':[True, False, True]}]),
           dict(method='update',
                label='Employment Density & Seating', visible=True,
                args=[{'label': 'Employment Density','visible':[False, True, True]}])
          ]

um_buttons = [{'active':0, 'showactive':True, 'buttons':buttons,
               'direction': 'down', 'xanchor': 'left','yanchor': 'bottom', 'x': 0.71, 'y': 1.01}]
map_annotations = [{'text':'Please select a map view to display', 'x': 1, 'y': 1.1,
                    'showarrow': False, 'font':{'family':'Arial','size':14}}]

fig.update_layout(updatemenus=um_buttons, annotations=map_annotations)

# Display the map
fig.show()

<div class="usecase-body">

**Our interactive map is now complete!**

</div>

<div class="usecase-body">

Now you can use the controls on the map above to explore the City of Melbourne and observe the residential density and employment density of each city block in relation to venue seating capacity.<br><br>

If you would like to extend this interactive map further, please visit the __[City of Melbourne Open Data Site](https://data.melbourne.vic.gov.au/)__ and explore some of the other valuable datasets including:
- __[Off Street Parking](https://data.melbourne.vic.gov.au/Transport/Off-street-car-parking-2020/g9am-cna5)__
- __[Pedestrian Counting System](https://data.melbourne.vic.gov.au/Transport/Pedestrian-Counting-System-Monthly-counts-per-hour/b2ak-trbp)__
- __[Microclimate sensor readings](https://data.melbourne.vic.gov.au/Environment/Microclimate-Sensor-Readings/u4vh-84j8?src=featured_banner)__

</div>

<div id="conclusion" class="usecase-unnumbered-heading"><b>Conclusion</b></div>


<div class="usecase-body">This usecase combined residential density, employment density and venue seating capacity into interactive geospatial visualisations for the City of Melbourne. The final map supports exploratory comparison of potential demand and existing hospitality capacity at CLUE block level.

This directly links back to the scenario: a cafe, restaurant or bar owner needs to identify where potential customers live or work and compare those areas with existing venue seating. By bringing residential density, employment density and venue seating together in one interactive map, the usecase helps the user investigate areas that may be suitable for a new venue or business expansion and supports a more informed location decision.</div>

<div class="usecase-body">

**What we achieved in this analysis**

We retrieved and prepared four CLUE datasets, aggregated residential and employment information by block, and combined these measures with venue seating on interactive maps.

**What we learned from this analysis**

Residential density, employment density and venue seating provide complementary views of the city. Interactive layer controls make it easier to compare these measures in the same geographic context.

**Observations for further opportunities**

The analysis is limited to the selected 2020 CLUE data. Additional variables such as pedestrian counts, rent, tourism, competitor type and more recent data could strengthen future business-location analysis.

</div>